# XAI Analysis: Multi-Model & Cross-Domain

**Scope:**
- **4 model families** (1 representative each): `tfidf_lr`, `cnn`, `distilbert`, `roberta`
- **3 single-source variants** per model: trained on `twitter`, `skytrax`, `airlinequality`
- **Per-sample (local) explanations:** 20 samples per class (stratified) = 60 samples per model
- **Global aggregate:** Top-K most influential words per predicted class (LIME aggregates)
- **Cross-domain XAI:** RoBERTa-twitter explained on twitter/skytrax/airlinequality test samples

**Methods:** SHAP (transformers) + LIME (all models).

## Setup (Google Drive)
Drive root: `My Drive/PATH/TO/`

Results saved to `results/xai/`.

**Runtime:** T4 ~1.5-2 hrs, L4/H100 ~30-45 min.

## 1. Setup

In [ ]:
!pip install -q transformers shap lime

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/PATH/TO"  # TODO: set to your Google Drive project path
SPLITS_DIR = f"{DRIVE_BASE}/data/splits"
MODELS_DIR = f"{DRIVE_BASE}/models"
XAI_DIR = f"{DRIVE_BASE}/results/xai"

import os
for sub in ["shap", "lime", "aggregate", "cross_domain"]:
    os.makedirs(f"{XAI_DIR}/{sub}", exist_ok=True)

assert os.path.isdir(SPLITS_DIR)
print("Paths OK.")

In [ ]:
import csv
import pickle
import random
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}
LABEL_NAMES = ["negative", "neutral", "positive"]
SINGLE_SOURCES = ["twitter", "skytrax", "airlinequality"]
N_SAMPLES_PER_CLASS = 20
TOP_K_GLOBAL = 20
LIME_NUM_SAMPLES = 500
PAD_IDX, UNK_IDX = 0, 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. DL Model Class (for CNN checkpoint)

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=300, num_classes=3, num_filters=128, kernel_sizes=(3,4,5), dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)
        outs = [torch.relu(c(x)).max(dim=2).values for c in self.convs]
        return self.fc(self.dropout(torch.cat(outs, dim=1)))

print("DL class defined.")

## 3. Unified predict_proba for LIME

In [ ]:
def load_test_split(source):
    path = f"{SPLITS_DIR}/{source}_test.csv"
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            texts.append(row["text"])
            labels.append(row["label"])
    return texts, labels


def stratified_samples(texts, labels, n_per_class, seed=SEED):
    rng = random.Random(seed)
    out = []
    for name in LABEL_NAMES:
        idxs = [i for i, l in enumerate(labels) if l == name]
        picked = rng.sample(idxs, min(n_per_class, len(idxs)))
        for i in picked:
            out.append({"text": texts[i], "true_label": name, "idx": i})
    return out


def build_predictor(model_kind, model_key, train_source):
    """Return a predict_proba(texts)->(N,3) function for any model family."""
    model_dir = f"{MODELS_DIR}/{model_key}/{train_source}"

    if model_kind == "tfidf":
        with open(f"{model_dir}/tfidf.pkl", "rb") as f: tfidf = pickle.load(f)
        with open(f"{model_dir}/model.pkl", "rb") as f: clf = pickle.load(f)
        def predict(texts):
            X = tfidf.transform(list(texts))
            if hasattr(clf, "predict_proba"):
                return clf.predict_proba(X)
            # LinearSVC: convert decision_function to pseudo-probs
            df = clf.decision_function(X)
            e = np.exp(df - df.max(axis=1, keepdims=True))
            return e / e.sum(axis=1, keepdims=True)
        return predict

    if model_kind == "dl":
        ckpt = torch.load(f"{model_dir}/model.pt", map_location=device, weights_only=False)
        word2idx = ckpt["word2idx"]
        max_len = ckpt["max_len"]
        model = TextCNN(ckpt["vocab_size"]).to(device)
        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()
        def predict(texts):
            seq = np.zeros((len(texts), max_len), dtype=np.int64)
            for i, t in enumerate(texts):
                for j, tok in enumerate(t.split()[:max_len]):
                    seq[i, j] = word2idx.get(tok, UNK_IDX)
            with torch.no_grad():
                logits = model(torch.from_numpy(seq).to(device))
                return torch.softmax(logits, dim=1).cpu().numpy()
        return predict

    # transformer
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()
    max_len = 128  # conservative for XAI (short perturbations)
    def predict(texts):
        probs = np.zeros((len(texts), 3))
        batch_size = 16
        for start in range(0, len(texts), batch_size):
            batch = list(texts[start:start+batch_size])
            enc = tokenizer(batch, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
            with torch.no_grad():
                logits = model(input_ids=enc["input_ids"].to(device),
                               attention_mask=enc["attention_mask"].to(device)).logits
                probs[start:start+batch_size] = torch.softmax(logits, dim=1).cpu().numpy()
        return probs
    return predict

print("Predictor builder ready.")

## 4. LIME: Local + Aggregate (4 models × 3 sources = 12 combos)

Per combination: 60 samples explained, local HTMLs saved, global top-K words aggregated per predicted class.

In [ ]:
from lime.lime_text import LimeTextExplainer

MODELS_FOR_XAI = [
    ("tfidf_lr",   "tfidf"),
    ("cnn",        "dl"),
    ("distilbert", "transformer"),
    ("roberta",    "transformer"),
]

lime_explainer = LimeTextExplainer(class_names=LABEL_NAMES, random_state=SEED)
all_local_rows = []
all_aggregate_rows = []

for model_key, kind in MODELS_FOR_XAI:
    for source in SINGLE_SOURCES:
        combo = f"{model_key}_{source}"
        print(f"\n=== LIME: {combo} ===")

        predict = build_predictor(kind, model_key, source)
        texts, labels = load_test_split(source)
        samples = stratified_samples(texts, labels, N_SAMPLES_PER_CLASS)

        # Per-class word-score aggregator (signed score for predicted class)
        per_class_word_scores = {n: defaultdict(list) for n in LABEL_NAMES}

        for i, s in enumerate(samples):
            probs = predict([s["text"]])[0]
            pred_class = int(np.argmax(probs))
            pred_label = LABEL_NAMES[pred_class]

            exp = lime_explainer.explain_instance(
                s["text"], predict, num_features=10,
                num_samples=LIME_NUM_SAMPLES, labels=[pred_class],
            )
            feats = exp.as_list(label=pred_class)

            # Save HTML (first 3 per class to keep file count sane)
            if i % N_SAMPLES_PER_CLASS < 3:
                html_dir = f"{XAI_DIR}/lime/{combo}"
                os.makedirs(html_dir, exist_ok=True)
                exp.save_to_file(f"{html_dir}/sample_{i:02d}_{s['true_label']}.html", labels=[pred_class])

            for word, score in feats:
                per_class_word_scores[pred_label][word.lower()].append(score)

            all_local_rows.append({
                "model": model_key,
                "train_source": source,
                "sample_idx": i,
                "true_label": s["true_label"],
                "predicted_label": pred_label,
                "text": s["text"][:200],
                "top_features": " | ".join(f"{w}({sc:+.3f})" for w, sc in feats),
            })

        # Aggregate: top-K words per predicted class (mean signed importance)
        for pred_label, words in per_class_word_scores.items():
            scored = [(w, np.mean(s), len(s)) for w, s in words.items() if len(s) >= 2]
            scored.sort(key=lambda x: abs(x[1]), reverse=True)
            for rank, (word, mean_score, count) in enumerate(scored[:TOP_K_GLOBAL]):
                all_aggregate_rows.append({
                    "model": model_key,
                    "train_source": source,
                    "predicted_class": pred_label,
                    "rank": rank + 1,
                    "word": word,
                    "mean_importance": mean_score,
                    "support": count,
                })
        print(f"  Processed {len(samples)} samples, saved HTMLs + aggregate")

        if torch.cuda.is_available(): torch.cuda.empty_cache()

# Save CSVs
with open(f"{XAI_DIR}/lime/lime_local.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(all_local_rows[0].keys()))
    w.writeheader(); w.writerows(all_local_rows)
with open(f"{XAI_DIR}/aggregate/lime_global_topk.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(all_aggregate_rows[0].keys()))
    w.writeheader(); w.writerows(all_aggregate_rows)
print(f"\nLIME: saved {len(all_local_rows)} local + {len(all_aggregate_rows)} aggregate rows")

## 5. SHAP: Transformer Models Only (DistilBERT, RoBERTa)

SHAP transformer pipelines are the canonical choice. We explain 15 samples/source for each (5/class).

In [ ]:
import shap
from transformers import pipeline

SHAP_N_PER_CLASS = 5
shap_local_rows = []

for model_key in ["distilbert", "roberta"]:
    for source in SINGLE_SOURCES:
        combo = f"{model_key}_{source}"
        print(f"\n=== SHAP: {combo} ===")

        model_dir = f"{MODELS_DIR}/{model_key}/{source}"
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
        model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        if torch.cuda.is_available(): model = model.cuda()
        model.eval()

        clf_pipeline = pipeline(
            "text-classification", model=model, tokenizer=tokenizer,
            device=0 if torch.cuda.is_available() else -1,
            top_k=None, truncation=True, max_length=128,
        )

        texts, labels = load_test_split(source)
        samples = stratified_samples(texts, labels, SHAP_N_PER_CLASS)
        sample_texts = [s["text"] for s in samples]

        explainer = shap.Explainer(clf_pipeline)
        shap_values = explainer(sample_texts)

        html_dir = f"{XAI_DIR}/shap/{combo}"
        os.makedirs(html_dir, exist_ok=True)

        for i, s in enumerate(samples):
            sv = shap_values[i]
            pred_class = int(np.argmax(sv.base_values + sv.values.sum(axis=0)))
            pred_label = LABEL_NAMES[pred_class]
            tok_scores = list(zip(sv.data, sv.values[:, pred_class]))
            tok_scores.sort(key=lambda x: x[1], reverse=True)

            # HTML viz
            html = shap.plots.text(sv, display=False) or ""
            with open(f"{html_dir}/sample_{i:02d}_{s['true_label']}.html", "w", encoding="utf-8") as f:
                f.write(f"<h3>True: {s['true_label']} | Pred: {pred_label}</h3><p>{s['text']}</p>{html}")

            shap_local_rows.append({
                "model": model_key,
                "train_source": source,
                "sample_idx": i,
                "true_label": s["true_label"],
                "predicted_label": pred_label,
                "text": s["text"][:200],
                "top_positive_tokens": " | ".join(f"{t.strip()}({v:+.3f})" for t, v in tok_scores[:5]),
                "top_negative_tokens": " | ".join(f"{t.strip()}({v:+.3f})" for t, v in tok_scores[-5:]),
            })

        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"  SHAP done for {combo} ({len(samples)} samples)")

with open(f"{XAI_DIR}/shap/shap_local.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(shap_local_rows[0].keys()))
    w.writeheader(); w.writerows(shap_local_rows)
print(f"\nSHAP: saved {len(shap_local_rows)} local rows")

## 6. Cross-Domain XAI

**Question:** RoBERTa-twitter was trained on tweets. When we apply it to skytrax/airlinequality test samples, which words does it latch onto? Does it misattribute because of domain shift?

Explains `roberta/twitter` (best in-domain) on 15 samples from each of twitter, skytrax, airlinequality test sets.

In [ ]:
cd_predict = build_predictor("transformer", "roberta", "twitter")
cd_rows = []

for test_source in SINGLE_SOURCES:
    print(f"\n=== Cross-Domain XAI: roberta-twitter on {test_source} ===")
    texts, labels = load_test_split(test_source)
    samples = stratified_samples(texts, labels, 5)

    html_dir = f"{XAI_DIR}/cross_domain/roberta_twitter_on_{test_source}"
    os.makedirs(html_dir, exist_ok=True)

    for i, s in enumerate(samples):
        probs = cd_predict([s["text"]])[0]
        pred_class = int(np.argmax(probs))
        pred_label = LABEL_NAMES[pred_class]

        exp = lime_explainer.explain_instance(
            s["text"], cd_predict, num_features=10,
            num_samples=LIME_NUM_SAMPLES, labels=[pred_class],
        )
        feats = exp.as_list(label=pred_class)
        exp.save_to_file(f"{html_dir}/sample_{i:02d}_{s['true_label']}.html", labels=[pred_class])

        cd_rows.append({
            "test_source": test_source,
            "sample_idx": i,
            "true_label": s["true_label"],
            "predicted_label": pred_label,
            "correct": s["true_label"] == pred_label,
            "text": s["text"][:200],
            "top_features": " | ".join(f"{w}({sc:+.3f})" for w, sc in feats),
        })

with open(f"{XAI_DIR}/cross_domain/cross_domain_xai.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(cd_rows[0].keys()))
    w.writeheader(); w.writerows(cd_rows)
print(f"\nCross-domain XAI: saved {len(cd_rows)} rows")

## 7. Summary

In [ ]:
print("=== XAI Artifacts ===")
print(f"LIME local ({len(all_local_rows)} rows):  {XAI_DIR}/lime/lime_local.csv")
print(f"LIME global ({len(all_aggregate_rows)} rows): {XAI_DIR}/aggregate/lime_global_topk.csv")
print(f"SHAP local ({len(shap_local_rows)} rows):  {XAI_DIR}/shap/shap_local.csv")
print(f"Cross-domain XAI ({len(cd_rows)} rows):   {XAI_DIR}/cross_domain/cross_domain_xai.csv")
print(f"\nHTMLs: {XAI_DIR}/{{lime,shap,cross_domain}}/<combo>/")
print("\nAll files on Drive — sync to your local machine.")